## IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

## IMPORTAR LOS DATOS

In [2]:
num = pd.read_pickle('../02_datos/03_Entrenamiento/num_resultado_calidad.pkl')
cat = pd.read_pickle('../02_datos/03_Entrenamiento/cat_resultado_calidad.pkl')

# Nuevas Variables

### Vamos a crear:
- Las identificadas en EDA
- Lags
- Ventanas móviles

In [8]:
df = pd.concat([cat, num], axis = 1 )
df 

,store_id,item_id,d,year,month,wday,weekday,event_name_1,event_type_1,ventas,sell_price,wm_yr_wk
date,,,,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National,0,1.250,11249
2013-01-01,CA_3,FOODS_3_120,d_704,2013,1,4,Tuesday,NewYear,National,33,1.250,11249
2013-01-01,CA_3,FOODS_3_202,d_704,2013,1,4,Tuesday,NewYear,National,0,4.980,11249
2013-01-01,CA_3,FOODS_3_252,d_704,2013,1,4,Tuesday,NewYear,National,0,4.980,11249
2013-01-01,CA_3,FOODS_3_288,d_704,2013,1,4,Tuesday,NewYear,National,20,4.280,11249
...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-30,CA_4,FOODS_3_329,d_1767,2015,11,3,Monday,Sin evento,Sin evento,9,1.680,11544
2015-11-30,CA_4,FOODS_3_555,d_1767,2015,11,3,Monday,Sin evento,Sin evento,26,2.480,11544
2015-11-30,CA_4,FOODS_3_586,d_1767,2015,11,3,Monday,Sin evento,Sin evento,13,2.480,11544


## Variable demanda intermitente

Esta variable va a identificar cuantos días seguidos han transcurrido con ventas cero

In [17]:
def rotura_stock(ventas, n = 5):
    cero_ventas = pd.Series(np.where(ventas == 0, 1, 0))
    num_ceros = cero_ventas.rolling(n).sum()
    rotura_stock = np.where(num_ceros == n, 1, 0)
    return (rotura_stock)

In [13]:
df = df.sort_values(by = ['store_id','item_id','date'])

In [18]:
df['rotura_stock_3'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 3)).values

In [20]:
df['rotura_stock_7'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 7)).values

In [23]:
df['rotura_stock_15'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 15)).values